# Next-day direction and range: a first look with LightGBM and a GRU

Phase 1 of [`docs/ml_plan.md`](../docs/ml_plan.md). Before anything goes into a package, this
notebook tries the whole method once, end to end:

| Section | Question |
|---|---|
| 1 Bars, targets, features | What does each model see at the close of day t, and what must it forecast for t+1? Does any feature look ahead? |
| 2 Walk-forward | How are the folds cut, and what is purged at their boundaries? |
| 3 Baselines and metrics | What does a model have to beat, and how is "beat" measured? |
| 4 LightGBM | One `binary` model for the direction; one `quantile` model per τ for the range. |
| 5 GRU | The same two targets from a 60-day window of the same features. |
| 6 Results and sanity checks | On the synthetic tickers, is the direction no better than chance, and does the range beat a constant-width band? |
| 7 The checks can fail | A planted signal must be found; a leaked future return must be caught; prediction at t must not change when later bars are removed. |

**Data.** The default is the six synthetic tickers (generated, offline, not market data). To run
on real prices, save them first with `save_all()` or the dashboard's **Download all**, then set
`DATA = 'local'` and list the symbols. **Do not commit this notebook with outputs**: run on saved
data, its tables and charts would be market data. Clear all outputs before committing.

**Needs** the `stats` extra (scipy), LightGBM and PyTorch: `pip install -e ".[stats]"`, then
`pip install lightgbm` and, for a CPU-only PyTorch,
`pip install torch --index-url https://download.pytorch.org/whl/cpu`. (From Phase 2, the `ml`
extra installs both.) The whole run takes about 15 minutes on four CPU cores, most of it the GRU;
`GRU_TICKERS` limits which tickers it runs on.

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e ".[stats]"`, so
# `from src.tools...` resolves from any working directory. On Colab, an editable install is only
# seen by a new interpreter, so the repo root goes on sys.path for this kernel.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}; edit REPO to point at your clone.')
    %pip install -q -e "{REPO}[stats]" lightgbm
    sys.path.insert(0, REPO)

In [ ]:
import math
import os
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from scipy import stats

from src.tools.price_return import read_local, synthetic_bars, synthetic_tickers
from src.tools.price_return.stats import stationary_bootstrap

try:
    import lightgbm as lgb
    import torch
except ImportError as exc:
    raise ImportError('This notebook needs LightGBM and PyTorch: pip install lightgbm, and '
                      'pip install torch --index-url https://download.pytorch.org/whl/cpu') from exc

print(f'LightGBM {lgb.__version__}, PyTorch {torch.__version__}')

## Parameters

The design choices confirmed in the plan (2026-10-02). `LONG` are the tickers the sanity checks
run on: SYN-LEV starts in August 2022, which leaves too few test days after a 3-year warm-up, so
it is shown but not judged.

In [ ]:
DATA = 'synthetic'                       # 'synthetic' (offline) or 'local' (saved Yahoo data)
TICKERS = synthetic_tickers() if DATA == 'synthetic' else ['ES=F', 'NQ=F', 'GC=F', 'CL=F']
LONG = [t for t in TICKERS if t != 'SYN-LEV']       # judged by the sanity checks
GRU_TICKERS = TICKERS                    # the GRU is the slow part; shorten this list to save time

TAUS = (0.05, 0.1587, 0.5, 0.8413, 0.95)  # quantiles of the next close: 90% and 68% intervals
FIRST_TRAIN = 756                         # 3 years of bars before the first test fold
STEP = 63                                 # test one quarter, then retrain (expanding window)
HORIZON = 1                               # next-day labels
WINDOW = 60                               # the GRU's look-back, in bars
N_BOOT, MEAN_BLOCK = 1000, 20             # stationary bootstrap for the comparisons
SEED = 0

## 1 Bars, targets, features

`C_t` is the close of bar t. The only forward-looking columns are the two labels: tomorrow's
direction `up = 1[C_{t+1} > C_t]` and tomorrow's return `r_{t+1}`, whose quantiles the range
models forecast. A return that spans a non-positive close (SYN-OIL on 2020-04-20) is dropped, as
`price_return` drops it.

Sixteen features, each from bar t and the bars before it: the return and four lags, its size,
20- and 60-day and EWMA volatility, Parkinson and Garman-Klass range volatility, the true range
over the close, the distance from the 20- and 50-day averages, a 14-day RSI, and the day of the
week.

In [ ]:
def load_bars(ticker):
    """Daily Open/High/Low/Close bars: generated for a synthetic ticker, else read from data/local."""
    if DATA == 'synthetic':
        return synthetic_bars(ticker)
    return read_local(ticker)


def returns(bars):
    """Close-to-close return of each bar, NaN where it spans a non-positive close."""
    c = bars['Close']
    return c.pct_change().where((c > 0) & (c.shift(1) > 0))


def features(bars):
    """The feature frame: every column at t uses bars up to t only."""
    o, h, l, c = (bars[k] for k in ('Open', 'High', 'Low', 'Close'))
    r = returns(bars)
    ok = (o > 0) & (h > 0) & (l > 0) & (c > 0)
    log_hl = np.log((h / l).where(ok))
    log_co = np.log((c / o).where(ok))
    gain, loss = r.clip(lower=0), (-r).clip(lower=0)
    rma = lambda x: x.ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
    prev_c = c.shift(1)
    true_range = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    f = pd.DataFrame(index=bars.index)
    for lag in range(5):
        f[f'ret_{lag}'] = r.shift(lag)
    f['abs_ret'] = r.abs()
    f['vol_20'] = r.rolling(20).std()
    f['vol_60'] = r.rolling(60).std()
    f['vol_ewma'] = np.sqrt((r ** 2).ewm(alpha=0.06, adjust=False, min_periods=20).mean())
    f['vol_parkinson'] = np.sqrt((log_hl ** 2).rolling(20).mean() / (4 * math.log(2)))
    f['vol_gk'] = np.sqrt((0.5 * log_hl ** 2 - (2 * math.log(2) - 1) * log_co ** 2).rolling(20).mean())
    f['tr_close'] = (true_range / c).where(ok & (prev_c > 0))
    f['dist_sma20'] = c / c.rolling(20).mean() - 1
    f['dist_sma50'] = c / c.rolling(50).mean() - 1
    f['rsi_14'] = 100 - 100 / (1 + rma(gain) / rma(loss))
    f['dow'] = bars.index.dayofweek.astype(float)
    return f


def dataset(bars):
    """Features, the two labels and the next date, on the rows where all are defined."""
    f = features(bars)
    r_next = returns(bars).shift(-1)
    d = f.assign(y_ret=r_next, y_up=(r_next > 0).astype(float), next_date=f.index.to_series().shift(-1))
    return d.dropna()


FEATURES = list(features(synthetic_bars('SYN-INDEX')).columns)
DATA_BY_TICKER = {t: dataset(load_bars(t)) for t in TICKERS}
pd.DataFrame({t: {'rows': len(d), 'from': d.index[0].date(), 'to': d.index[-1].date(),
                  'up share': f"{d['y_up'].mean():.1%}"} for t, d in DATA_BY_TICKER.items()}).T

**No look-ahead.** The test `ta_tools` uses for its indicators: cut the bars after t, recompute,
and every feature at or before t must be unchanged. A feature that peeked at a later bar would
change.

In [ ]:
def check_no_lookahead(bars, cuts=(300, 1200, 2000)):
    full = features(bars)
    for cut in cuts:
        part = features(bars.iloc[:cut + 1])
        pd.testing.assert_frame_equal(part, full.iloc[:cut + 1], check_exact=False, rtol=1e-12)
    return f'{len(FEATURES)} features unchanged at {len(cuts)} cut points'

check_no_lookahead(synthetic_bars('SYN-OIL'))   # the ticker with the negative settle

## 2 Walk-forward

Expanding window: the first fold trains on the first 756 rows and tests the next 63; each later
fold adds 63 rows to the training window. A training row whose label interval (t, t+h] reaches the
first test row is purged: h - 1 rows, so none for the next-day labels. The early-stopping tail
inside each training window is purged from the rows before it the same way.

In [ ]:
def walk_forward(n, first_train=FIRST_TRAIN, step=STEP, horizon=HORIZON):
    """(train_rows, test_rows) position ranges, the training rows purged of label overlap."""
    folds, a = [], first_train
    while a < n:
        b = min(a + step, n)
        folds.append((np.arange(0, a - (horizon - 1)), np.arange(a, b)))
        a = b
    return folds


def purged_split(train, horizon=HORIZON, tail=0.2):
    """Fit and early-stopping rows inside a training window, the last `tail` held out."""
    cut = int(len(train) * (1 - tail))
    return train[:cut - (horizon - 1)], train[cut:]


for h in (1, 5):
    tr, te = walk_forward(3000, horizon=h)[0]
    labels_end = tr[-1] + h                      # the last training label's bar
    print(f'h = {h}: last training row {tr[-1]}, its label at {labels_end}, first test row {te[0]}'
          f' -> {te[0] - 1 - tr[-1]} purged, overlap: {labels_end > te[0]}')
print(f"SYN-INDEX: {len(walk_forward(len(DATA_BY_TICKER['SYN-INDEX'])))} folds")

## 3 Baselines and metrics

Each baseline is refitted per fold from the training rows, or uses only what is known at t:

- **Direction:** the base rate, the up-share of the training labels.
- **Constant width:** the empirical quantiles of the training window's returns.
- **`price_range`:** a lognormal ±zσ band, with σ the trailing 250-day standard deviation, as
  `options.price_range` is fed by `latest_snapshot`.
- **20-day σ:** the training mean plus z times the 20-day standard deviation.
- **EWMA σ × standardized quantiles:** EWMA volatility at t times the training quantiles of
  `r_{t+1} / σ_t`. The plan found this the band to beat.

Metrics: log loss and Brier (with Murphy's decomposition) for the direction; coverage, Kupiec,
Christoffersen and pinball loss for the range. Differences are compared with the stationary
bootstrap from `price_return.stats`, which keeps blocks of days together.

In [ ]:
Z = {tau: stats.norm.ppf(tau) for tau in TAUS}


def baselines(d, folds):
    """Out-of-sample baseline forecasts for every test row: base rate and four quantile bands."""
    sig250 = d['ret_0'].rolling(250).std()          # rows are consecutive bars here
    out = []
    for train, test in folds:
        tr, te = d.iloc[train], d.iloc[test]
        z_std = tr['y_ret'] / tr['vol_ewma']
        f = pd.DataFrame(index=te.index)
        f['p_base'] = tr['y_up'].mean()
        for tau in TAUS:
            f[('constant', tau)] = tr['y_ret'].quantile(tau)
            f[('price_range', tau)] = np.exp(Z[tau] * sig250.iloc[test].fillna(tr['y_ret'].std())) - 1
            f[('vol_20', tau)] = tr['y_ret'].mean() + Z[tau] * te['vol_20']
            f[('ewma_std_q', tau)] = te['vol_ewma'] * z_std.quantile(tau)
        out.append(f)
    return pd.concat(out)


def log_loss(p, y):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def murphy(p, y, bins=10):
    """Brier = reliability - resolution + uncertainty, over equal-width forecast bins."""
    g = pd.DataFrame({'p': p, 'y': y}).groupby(pd.cut(p, np.linspace(0, 1, bins + 1), include_lowest=True),
                                                 observed=True)
    n, ybar = len(y), np.mean(y)
    rel = sum(len(x) * (x['p'].mean() - x['y'].mean()) ** 2 for _, x in g) / n
    res = sum(len(x) * (x['y'].mean() - ybar) ** 2 for _, x in g) / n
    return {'brier': float(np.mean((p - y) ** 2)), 'reliability': rel, 'resolution': res,
            'uncertainty': ybar * (1 - ybar)}


def pinball(q, y, tau):
    u = y - q
    return np.maximum(tau * u, (tau - 1) * u)


def kupiec_p(miss, rate):
    n, x = len(miss), int(miss.sum())
    ph = x / n
    lr = -2 * ((n - x) * math.log(1 - rate) + x * math.log(rate)
               - (n - x) * math.log(1 - ph) - x * math.log(ph))
    return 1 - stats.chi2.cdf(lr, 1)


def christoffersen(miss):
    """Independence test of the misses: p-value, and P(miss | miss yesterday) / P(miss | hit)."""
    m = np.asarray(miss, dtype=int)
    a, b = m[:-1], m[1:]
    n00, n01 = np.sum((a == 0) & (b == 0)), np.sum((a == 0) & (b == 1))
    n10, n11 = np.sum((a == 1) & (b == 0)), np.sum((a == 1) & (b == 1))
    p01, p11, p = n01 / (n00 + n01), n11 / (n10 + n11), (n01 + n11) / len(a)
    l0 = (n00 + n10) * math.log(1 - p) + (n01 + n11) * math.log(p)
    l1 = (n00 * math.log(1 - p01) + n01 * math.log(p01)
          + n10 * math.log(1 - p11) + n11 * math.log(p11))
    return 1 - stats.chi2.cdf(-2 * (l0 - l1), 1), p11 / p01


def boot_means(x, seed=SEED):
    """Stationary-bootstrap draws of the mean of a daily series."""
    x = np.asarray(x)
    return x[stationary_bootstrap(len(x), MEAN_BLOCK, N_BOOT, seed)].mean(axis=1)


def range_scores(q, y):
    """Coverage, tests and mean pinball loss of a {tau: forecast} band."""
    q = {tau: np.asarray(q[tau]) for tau in TAUS}
    miss68 = (y < q[0.1587]) | (y > q[0.8413])
    miss90 = (y < q[0.05]) | (y > q[0.95])
    p_ind, ratio = christoffersen(miss68)
    daily = np.mean([pinball(q[tau], y, tau) for tau in TAUS], axis=0)
    return {'cover68': 1 - miss68.mean(), 'cover90': 1 - miss90.mean(),
            'kupiec_p': kupiec_p(miss68, 1 - 0.6826), 'christoffersen_p': p_ind,
            'cluster_ratio': ratio, 'pinball_bp': daily.mean() * 1e4, 'width68': np.mean(q[0.8413] - q[0.1587])}, daily

## 4 LightGBM

Small and regularized: 15 leaves, at least 50 rows a leaf, learning rate 0.03, up to 300 trees.
The range models forecast quantiles of the **standardized** return `r_{t+1} / σ_t`, with σ_t the
EWMA volatility known at t, and multiply back by σ_t. Fitted to raw returns instead, they
under-covered (about 65% for the nominal 68% interval) and gained less over the constant band:
the trees spend their splits rediscovering the volatility scale, which dividing by σ_t supplies.
Each fold early-stops on the last 20% of its training window, then refits on the whole window
with the best number of trees. `deterministic=True` and one thread, so a rerun gives the same
forecasts. Crossed quantiles are sorted.

In [ ]:
LGB_PARAMS = dict(num_leaves=15, min_data_in_leaf=50, learning_rate=0.03, feature_fraction=0.8,
                  lambda_l2=1.0, deterministic=True, num_threads=1, seed=SEED, verbose=-1)


def lgb_fit_predict(objective, X, y, train, test, **extra):
    params = dict(LGB_PARAMS, objective=objective, **extra)
    fit, val = purged_split(train)
    probe = lgb.train(params, lgb.Dataset(X[fit], y[fit]), num_boost_round=300,
                      valid_sets=[lgb.Dataset(X[val], y[val])],
                      callbacks=[lgb.early_stopping(30, verbose=False)])
    rounds = max(probe.best_iteration, 1)
    model = lgb.train(params, lgb.Dataset(X[train], y[train]), num_boost_round=rounds)
    return model.predict(X[test]), rounds, model


def lgb_walk_forward(d, folds):
    X, y_up = d[FEATURES].to_numpy(), d['y_up'].to_numpy()
    sigma = d['vol_ewma'].to_numpy()
    y_std = d['y_ret'].to_numpy() / sigma                # standardized by the volatility known at t
    out, rounds, importance = [], [], []
    for train, test in folds:
        f = pd.DataFrame(index=d.index[test])
        f['p_model'], n, model = lgb_fit_predict('binary', X, y_up, train, test)
        rounds.append(n)
        importance.append(model.feature_importance('gain'))
        qs = np.column_stack([lgb_fit_predict('quantile', X, y_std, train, test, alpha=tau)[0] for tau in TAUS])
        qs.sort(axis=1)                          # no crossed quantiles
        qs *= sigma[test][:, None]               # back to returns
        for i, tau in enumerate(TAUS):
            f[('model', tau)] = qs[:, i]
        out.append(f)
    return pd.concat(out), rounds, pd.Series(np.mean(importance, axis=0), FEATURES)


LGB, LGB_INFO = {}, {}
t0 = time.time()
for t, d in DATA_BY_TICKER.items():
    folds = walk_forward(len(d))
    pred, rounds, imp = lgb_walk_forward(d, folds)
    LGB[t] = baselines(d, folds).join(pred)
    LGB_INFO[t] = {'folds': len(folds), 'median trees (direction)': int(np.median(rounds)),
                   'top features (direction)': ', '.join(imp.nlargest(3).index)}
print(f'LightGBM walk-forward on {len(TICKERS)} tickers: {time.time() - t0:.0f} s')
pd.DataFrame(LGB_INFO).T

## 5 GRU

One GRU layer of 32 units reads the last 60 rows of the 16 features, standardized with the
training window's means and standard deviations. The direction network ends in one logit
(binary cross-entropy); the range network in five quantiles (pinball loss) of the same
standardized return as LightGBM's, scaled back by σ_t. Adam with a small weight decay,
batches of 128, up to 40 epochs, early stopping on the purged last 20% of the training window
with a patience of 5, seeded per fold, on CPU with deterministic algorithms.

In [ ]:
torch.use_deterministic_algorithms(True)
torch.set_num_threads(max(1, os.cpu_count() or 1))


class GRUNet(torch.nn.Module):
    def __init__(self, n_features, n_out, hidden=32):
        super().__init__()
        self.gru = torch.nn.GRU(n_features, hidden, batch_first=True)
        self.head = torch.nn.Linear(hidden, n_out)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1])


def quantile_loss(pred, y, taus):
    u = y.unsqueeze(1) - pred
    return torch.maximum(taus * u, (taus - 1) * u).mean()


def windows(X, rows):
    """For each row, the WINDOW rows ending at it: (len(rows), WINDOW, n_features)."""
    idx = rows[:, None] + np.arange(-WINDOW + 1, 1)[None, :]
    return X[idx]


def gru_fit_predict(kind, X, y, train, test, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    train = train[train >= WINDOW - 1]
    mu, sd = X[train].mean(axis=0), X[train].std(axis=0) + 1e-12
    Xs = ((X - mu) / sd).astype(np.float32)
    scale = 1.0 if kind == 'direction' else float(np.std(y[train]))
    ys = (y / scale).astype(np.float32)
    fit, val = purged_split(train)
    net = GRUNet(X.shape[1], 1 if kind == 'direction' else len(TAUS))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-4)
    taus = torch.tensor(TAUS, dtype=torch.float32)
    loss_fn = ((lambda p, t: torch.nn.functional.binary_cross_entropy_with_logits(p[:, 0], t))
               if kind == 'direction' else (lambda p, t: quantile_loss(p, t, taus)))
    xv, yv = torch.from_numpy(windows(Xs, val)), torch.from_numpy(ys[val])
    best, best_state, wait = np.inf, None, 0
    for epoch in range(40):
        net.train()
        for batch in np.array_split(rng.permutation(fit), max(1, len(fit) // 128)):
            opt.zero_grad()
            loss = loss_fn(net(torch.from_numpy(windows(Xs, batch))), torch.from_numpy(ys[batch]))
            loss.backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            v = loss_fn(net(xv), yv).item()
        if v < best - 1e-6:
            best, best_state, wait = v, {k: x.clone() for k, x in net.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= 5:
                break
    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        out = net(torch.from_numpy(windows(Xs, test))).numpy()
    if kind == 'direction':
        return 1 / (1 + np.exp(-out[:, 0]))
    return np.sort(out, axis=1) * scale


def gru_walk_forward(d, folds):
    X, y_up = d[FEATURES].to_numpy(), d['y_up'].to_numpy()
    sigma = d['vol_ewma'].to_numpy()
    y_std = d['y_ret'].to_numpy() / sigma
    out = []
    for k, (train, test) in enumerate(folds):
        f = pd.DataFrame(index=d.index[test])
        f['p_model'] = gru_fit_predict('direction', X, y_up, train, test, SEED + k)
        qs = gru_fit_predict('range', X, y_std, train, test, SEED + k) * sigma[test][:, None]
        for i, tau in enumerate(TAUS):
            f[('model', tau)] = qs[:, i]
        out.append(f)
    return pd.concat(out)


GRU = {}
t0 = time.time()
for t in GRU_TICKERS:
    d = DATA_BY_TICKER[t]
    folds = walk_forward(len(d))
    GRU[t] = baselines(d, folds).join(gru_walk_forward(d, folds))
    print(f'{t}: {time.time() - t0:.0f} s')

## 6 Results and sanity checks

**Direction**: log loss and Brier of the model against the base rate, with the bootstrap interval
of `log loss(base) - log loss(model)` (positive means the model is better). On the synthetic
tickers the sign of tomorrow's return is independent of the past by construction, so the model
must not be better than the base rate: the plan's check is that the 99% interval does not lie
wholly above zero, and the mean gain is at most 0.002.

**Range**: coverage, the two tests and pinball loss for the model and every baseline. The model
must have a lower pinball loss than the constant band on each long ticker, a pooled gain whose
95% interval lies wholly above zero, and less miss clustering than the constant band.

In [ ]:
def direction_table(results, y_by_ticker):
    rows = {}
    for t, f in results.items():
        y = y_by_ticker[t].loc[f.index, 'y_up'].to_numpy()
        gain = log_loss(f['p_base'].to_numpy(), y) - log_loss(f['p_model'].to_numpy(), y)
        draws = boot_means(gain)
        lo, hi = np.quantile(draws, [0.005, 0.995])
        rows[t] = {'days': len(y), 'log loss base': log_loss(f['p_base'], y).mean(),
                   'log loss model': log_loss(f['p_model'], y).mean(),
                   'Brier base': murphy(f['p_base'].to_numpy(), y)['brier'],
                   'Brier model': murphy(f['p_model'].to_numpy(), y)['brier'],
                   'gain': gain.mean(), 'gain 99% low': lo, 'gain 99% high': hi,
                   'forecast spread': f['p_model'].std(),
                   'chance (pass)': (t not in LONG) or (lo <= 0 and gain.mean() <= 0.002)}
    return pd.DataFrame(rows).T


def range_table(results, y_by_ticker):
    rows, daily = [], {}
    for t, f in results.items():
        y = y_by_ticker[t].loc[f.index, 'y_ret'].to_numpy()
        for band in ('constant', 'price_range', 'vol_20', 'ewma_std_q', 'model'):
            s, pb = range_scores({tau: f[(band, tau)] for tau in TAUS}, y)
            rows.append(dict(ticker=t, band=band, **s))
            daily[(t, band)] = pb
    table = pd.DataFrame(rows).set_index(['ticker', 'band'])
    return table, daily


def range_check(table, daily):
    """The plan's range checks: per ticker, and the pooled relative gain over the constant band."""
    out, pooled = {}, []
    for t in [t for t in table.index.get_level_values(0).unique() if t in LONG]:
        const, model = table.loc[(t, 'constant')], table.loc[(t, 'model')]
        rel = (daily[(t, 'constant')] - daily[(t, 'model')]) / daily[(t, 'constant')].mean()
        pooled.append(boot_means(rel))
        out[t] = {'gain vs constant': rel.mean(), 'lower pinball': model['pinball_bp'] < const['pinball_bp'],
                  'less clustering': model['cluster_ratio'] < const['cluster_ratio']}
    draws = np.mean(pooled, axis=0)
    lo, hi = np.quantile(draws, [0.025, 0.975])
    passed = all(v['lower pinball'] and v['less clustering'] for v in out.values()) and lo > 0
    return pd.DataFrame(out).T, (float(draws.mean()), lo, hi), passed


fmt = {'log loss base': '{:.4f}', 'log loss model': '{:.4f}', 'Brier base': '{:.4f}',
       'Brier model': '{:.4f}', 'gain': '{:+.4f}', 'gain 99% low': '{:+.4f}',
       'gain 99% high': '{:+.4f}', 'forecast spread': '{:.4f}'}
rfmt = {'cover68': '{:.1%}', 'cover90': '{:.1%}', 'kupiec_p': '{:.3f}', 'christoffersen_p': '{:.2g}',
        'cluster_ratio': '{:.2f}', 'pinball_bp': '{:.2f}', 'width68': '{:.4f}'}

### LightGBM

In [ ]:
LGB_DIR = direction_table(LGB, DATA_BY_TICKER)
display(LGB_DIR.style.format(fmt))
LGB_RANGE, LGB_DAILY = range_table(LGB, DATA_BY_TICKER)
display(LGB_RANGE.style.format(rfmt))
LGB_CHECK, LGB_POOLED, LGB_RANGE_PASS = range_check(LGB_RANGE, LGB_DAILY)
display(LGB_CHECK)
print(f'LightGBM direction no better than chance on every long ticker: {LGB_DIR["chance (pass)"].all()}')
print(f'LightGBM range pooled gain over the constant band: {LGB_POOLED[0]:.1%} '
      f'(95% interval {LGB_POOLED[1]:.1%} to {LGB_POOLED[2]:.1%}); range checks pass: {LGB_RANGE_PASS}')

### GRU

In [ ]:
GRU_DIR = direction_table(GRU, DATA_BY_TICKER)
display(GRU_DIR.style.format(fmt))
GRU_RANGE, GRU_DAILY = range_table(GRU, DATA_BY_TICKER)
display(GRU_RANGE.style.format(rfmt))
GRU_CHECK, GRU_POOLED, GRU_RANGE_PASS = range_check(GRU_RANGE, GRU_DAILY)
display(GRU_CHECK)
print(f'GRU direction no better than chance on every long ticker: {GRU_DIR["chance (pass)"].all()}')
print(f'GRU range pooled gain over the constant band: {GRU_POOLED[0]:.1%} '
      f'(95% interval {GRU_POOLED[1]:.1%} to {GRU_POOLED[2]:.1%}); range checks pass: {GRU_RANGE_PASS}')

### Charts

Reliability of the direction forecasts (marker area grows with the number of days in the bin),
and the pinball loss of every band per ticker, relative to the constant band.

In [ ]:
def plot_reliability(results_by_model, ticker):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0.3, 0.7], y=[0.3, 0.7], mode='lines', name='perfect calibration',
                             line=dict(color='#8a8984', dash='dot')))
    y = DATA_BY_TICKER[ticker]
    for (name, res), color in zip(results_by_model.items(), ('#2a78d6', '#eb6834', '#1baf7a')):
        f = res[ticker]
        yy = y.loc[f.index, 'y_up']
        for col, label in (('p_model', name), ('p_base', 'base rate')):
            if col == 'p_base' and name != list(results_by_model)[0]:
                continue
            g = pd.DataFrame({'p': f[col], 'y': yy}).groupby(pd.cut(f[col], np.linspace(0, 1, 41)), observed=True)
            agg = g.agg(p=('p', 'mean'), y=('y', 'mean'), n=('y', 'size'))
            fig.add_trace(go.Scatter(x=agg['p'], y=agg['y'], mode='markers', name=label,
                                     marker=dict(size=np.clip(np.sqrt(agg['n']), 5, 40),
                                                 color='#52514e' if col == 'p_base' else color,
                                                 line=dict(color='white', width=1))))
    fig.update_layout(title=f'{ticker}: reliability of the direction forecasts', width=700, height=560,
                      xaxis=dict(title='forecast probability of an up day', range=[0.3, 0.7], tickformat='.0%'),
                      yaxis=dict(title='observed up frequency', range=[0.3, 0.7], tickformat='.0%'),
                      template='plotly_white')
    return fig


def plot_relative_pinball(tables):
    fig = make_subplots(rows=1, cols=len(tables), shared_yaxes=True, subplot_titles=list(tables))
    colors = {'price_range': '#eda100', 'vol_20': '#1baf7a', 'ewma_std_q': '#2a78d6', 'model': '#eb6834'}
    for i, table in enumerate(tables.values(), start=1):
        for band, color in colors.items():
            rel = [(table.loc[(t, band), 'pinball_bp'] / table.loc[(t, 'constant'), 'pinball_bp'] - 1) * 100
                   for t in table.index.get_level_values(0).unique()]
            fig.add_bar(x=list(table.index.get_level_values(0).unique()), y=rel, name=band,
                        marker_color=color, showlegend=(i == 1), row=1, col=i)
    fig.update_yaxes(title_text='pinball loss vs constant band (%)', ticksuffix='%', row=1, col=1)
    fig.update_layout(title='Range: mean pinball loss relative to the constant-width band (lower is better)',
                      barmode='group', width=1100, height=480, template='plotly_white')
    return fig


plot_reliability({'LightGBM': LGB, 'GRU': GRU}, TICKERS[0]).show()
plot_relative_pinball({'LightGBM': LGB_RANGE, 'GRU': GRU_RANGE}).show()

## 7 The checks can fail

A check that cannot fail proves nothing. A model that always predicted the base rate would pass
the direction check trivially, so three controls:

1. **A planted signal.** SYN-INDEX with each day's sign made to repeat yesterday's 58% of the
   time (magnitudes and volatility kept). The direction model must now beat the base rate, with a
   99% interval wholly above zero.
2. **A leaked future return.** `r_{t+1}` added as a feature. The direction check must fail.
3. **Prediction at t cannot see past t.** Remove every bar after a test date, rerun that fold,
   and the forecast for that date must be identical.

In [ ]:
def planted_bars(ticker='SYN-INDEX', repeat=0.58, seed=1):
    """`ticker`'s bars with signs re-drawn so that each repeats the previous one `repeat` of the time."""
    bars = synthetic_bars(ticker)
    r = bars['Close'].pct_change().fillna(0).to_numpy()
    rng = np.random.default_rng(seed)
    sign = np.where(r >= 0, 1.0, -1.0)
    for t in range(1, len(r)):
        sign[t] = sign[t - 1] if rng.random() < repeat else -sign[t - 1]
    new_close = bars['Close'].iloc[0] * np.cumprod(1 + np.abs(r) * sign)
    factor = new_close / bars['Close'].to_numpy()
    out = bars.copy()
    for col in ('Open', 'High', 'Low', 'Close'):
        out[col] = bars[col] * factor
    return out


def direction_only(d):
    folds = walk_forward(len(d))
    X, y = d[FEATURES].to_numpy(), d['y_up'].to_numpy()
    f = baselines(d, folds)[['p_base']]
    f['p_model'] = np.concatenate([lgb_fit_predict('binary', X, y, tr, te)[0] for tr, te in folds])
    return f


planted = dataset(planted_bars())
leaked = DATA_BY_TICKER['SYN-INDEX'].copy()
leaked['leak'] = leaked['y_ret']
FEATURES_LEAK = FEATURES + ['leak']

controls = {'planted signal': direction_only(planted)}
_saved = FEATURES
FEATURES = FEATURES_LEAK
controls['leaked r(t+1)'] = direction_only(leaked)
FEATURES = _saved
CONTROL = direction_table(controls, {'planted signal': planted, 'leaked r(t+1)': leaked})
CONTROL['must'] = ['beat the base rate', 'beat the base rate (so the check fails)']
CONTROL['beats base rate'] = CONTROL['gain 99% low'] > 0
display(CONTROL[['days', 'log loss base', 'log loss model', 'gain', 'gain 99% low', 'gain 99% high',
                 'beats base rate', 'must']].style.format(fmt))

In [ ]:
def forecast_at(bars, date):
    """LightGBM direction and median forecasts for `date`, from the fold that tests it."""
    d = dataset(bars)
    X = d[FEATURES].to_numpy()
    for train, test in walk_forward(len(d)):
        if date in d.index[test]:
            i = np.searchsorted(d.index[test], date)
            p = lgb_fit_predict('binary', X, d['y_up'].to_numpy(), train, test)[0][i]
            y_std = (d['y_ret'] / d['vol_ewma']).to_numpy()
            q = lgb_fit_predict('quantile', X, y_std, train, test, alpha=0.5)[0][i] * d['vol_ewma'].iloc[test[i]]
            return p, q
    raise ValueError(date)


bars = synthetic_bars('SYN-INDEX')
d = DATA_BY_TICKER['SYN-INDEX']
for date in d.index[[900, 1700, 2400]]:
    full = forecast_at(bars, date)
    # the bars up to `date` and one more: the label of `date` needs the next close, nothing later
    cut = forecast_at(bars.iloc[:bars.index.get_loc(date) + 2], date)
    print(f'{date:%Y-%m-%d}: full history {full[0]:.6f} / {full[1]:+.6f}, '
          f'later bars removed {cut[0]:.6f} / {cut[1]:+.6f}, identical: {full == cut}')

## Summary

The printout below is what Phase 1 set out to confirm before the method is written into a
package (Phase 2 onward).

In [ ]:
summary = pd.DataFrame({
    'LightGBM': {'direction no better than chance': bool(LGB_DIR['chance (pass)'].all()),
                 'range beats the constant band': LGB_RANGE_PASS,
                 'pooled pinball gain': f'{LGB_POOLED[0]:.1%} ({LGB_POOLED[1]:.1%} to {LGB_POOLED[2]:.1%})'},
    'GRU': {'direction no better than chance': bool(GRU_DIR['chance (pass)'].all()),
            'range beats the constant band': GRU_RANGE_PASS,
            'pooled pinball gain': f'{GRU_POOLED[0]:.1%} ({GRU_POOLED[1]:.1%} to {GRU_POOLED[2]:.1%})'},
})
display(summary)
print(f"Controls (LightGBM): planted signal found: {bool(CONTROL.loc['planted signal', 'beats base rate'])}, "
      f"leak caught: {bool(CONTROL.loc['leaked r(t+1)', 'beats base rate'])}")